In [1]:
import tensorflow as tf
import numpy as np
import glob
import json
import os

2025-04-19 14:59:01.222994: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-19 14:59:01.232434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745071141.241661 1694865 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745071141.244280 1694865 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-19 14:59:01.255489: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [32]:
def calculate_time_axis_stats(segment_length):
    """Calculate statistics for the time dimension (axis=1) of mel spectrograms"""
    
    # Get all TFRecord files for this segment length
    data_dir = f"/home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_{segment_length}s"
    tfrecord_files = tf.io.gfile.glob(f"{data_dir}/train/*.tfrecord")
    
    if not tfrecord_files:
        raise ValueError(f"No TFRecord files found for {segment_length}s segments")
    
    time_dimensions = []
    
    def parse_example(example):
        feature_description = {
            'mel_spectrogram': tf.io.FixedLenFeature([], tf.string),
            'labels': tf.io.VarLenFeature(tf.int64),
            'song_name': tf.io.FixedLenFeature([], tf.string),
            'segment_idx': tf.io.FixedLenFeature([], tf.int64),
            'total_segments': tf.io.FixedLenFeature([], tf.int64),
        }
        parsed = tf.io.parse_single_example(example, feature_description)
        mel_spec = tf.io.parse_tensor(parsed['mel_spectrogram'], out_type=tf.float32)
        
        if tf.rank(mel_spec) != 2:
            mel_spec = tf.reshape(mel_spec, [80, -1])
        
        return mel_spec
    
    for file_path in tfrecord_files:
        print(f"Processing {file_path}")
        dataset = tf.data.TFRecordDataset(file_path)
        dataset = dataset.map(parse_example)
        
        # Extract the time dimension length from each example
        for mel_spec in dataset:
            shape = tf.shape(mel_spec).numpy()
            time_dim = shape[1]  # Time is the axis=1
            time_dimensions.append(time_dim)
    
    # Calculate statistics
    time_dimensions = np.array(time_dimensions)
    stats = {
        "min": int(np.min(time_dimensions)),
        "max": int(np.max(time_dimensions)),
        "mean": int(np.mean(time_dimensions)),
        "median": int(np.median(time_dimensions)),
        "std": float(np.std(time_dimensions)),
        "count": int(len(time_dimensions))
    }

    json_dir = "../json/average_time_axis"
    os.makedirs(json_dir, exist_ok=True)
        
    with open(f"{json_dir}/average_time_axis_{segment_length}s.json", "w") as f:
        json.dump(stats, f, indent=2)
    
    print(f"Statistics for {segment_length}s segments saved to average_time_axis_{segment_length}s.json")
    return stats

In [30]:
segment_lengths = [3, 6, 10, 14, 20, 30]
results = {}
    
for length in segment_lengths:
    print(f"Processing {length}s segments...")
    stats = calculate_time_axis_stats(length)
    results[str(length)] = stats

Processing 3s segments...
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0026.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0144.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0133.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0110.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0213.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0028.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0137.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0194.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0073.tfrecord
Processing /home/georgios/Music Analys

2025-04-19 15:21:20.880335: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0234.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0176.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0159.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0109.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0025.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0042.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0111.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0024.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_3s/train/train_0205.tfrecord
Processing /home/georgios/Music Analysis/dataset_creation/tfreco

In [31]:
results

{'3': {'min': 108,
  'max': 162,
  'mean': 130,
  'median': 130,
  'std': 7.755777297292982,
  'count': 239758},
 '6': {'min': 216,
  'max': 323,
  'mean': 259,
  'median': 259,
  'std': 15.415232832295706,
  'count': 118592},
 '10': {'min': 359,
  'max': 539,
  'mean': 432,
  'median': 431,
  'std': 25.664076947265468,
  'count': 70078},
 '14': {'min': 503,
  'max': 754,
  'mean': 605,
  'median': 603,
  'std': 36.042277763738426,
  'count': 49330},
 '20': {'min': 718,
  'max': 1077,
  'mean': 865,
  'median': 862,
  'std': 51.27503675282696,
  'count': 33720},
 '30': {'min': 1077,
  'max': 1615,
  'mean': 1296,
  'median': 1292,
  'std': 77.39001106847496,
  'count': 21636}}